In [ ]:
# Imports
import tensorflow as tf

import cv2
import numpy as np
import matplotlib.pyplot as plt

# For extracting file
import os
import gzip
import shutil
from pathlib import Path

In [ ]:
# Essential functions

for gz_file in Path('../global-datasets/tensorflow-number-datasets').glob('*.gz'):
  output_file = gz_file.with_suffix("")
  # print(gz_file, output_file)

  with gzip.open(gz_file, 'rb') as file_in:
    with open(output_file, 'wb') as file_out:
      shutil.copyfileobj(file_in, file_out)
  
  print("Unzip: ", output_file)

def load_image(file_name):
  with open(file_name, 'rb') as f:
    magic_number = int.from_bytes(f.read(4), "big")
    image_count = int.from_bytes(f.read(4), "big")
    image_row = int.from_bytes(f.read(4), "big")
    image_col = int.from_bytes(f.read(4), "big")
    image_data = f.read()

    image = np.frombuffer(
      image_data, dtype=np.uint8
    ).reshape(
      image_count, image_row, image_col
    )
  
  return image

def load_label(file_name):
  with open(file_name, 'rb') as f:
    magic_number = int.from_bytes(f.read(4), 'big')
    count = int.from_bytes(f.read(4), 'big')
    label_data = f.read(count)
    
    label = np.frombuffer(
      label_data, dtype=np.uint8
    )

  return label

In [ ]:
# x is data/image
# y is label

num_class = 10

x_train = load_image("../global-datasets/tensorflow-number-datasets/train-images-idx3-ubyte")
y_train = load_label("../global-datasets/tensorflow-number-datasets/train-labels-idx1-ubyte")

x_test = load_image("../global-datasets/tensorflow-number-datasets/t10k-images-idx3-ubyte")
y_test = load_label("../global-datasets/tensorflow-number-datasets/t10k-labels-idx1-ubyte")

# Normalization to make the data value between 0 - 1
x_train = x_train.astype("uint8") / 255
x_test = x_test.astype("uint8") / 255

y_train = tf.keras.utils.to_categorical(y_train, num_class)
y_test = tf.keras.utils.to_categorical(y_test, num_class)

In [ ]:
# CNN Example

input_shape = (28, 28, 1)

# Create sequential Model (Stacking)
model = tf.keras.Sequential()

# Recevie Input (28 x 28 x 1)
model.add(tf.keras.Input(shape=input_shape))

# Conv 2D
model.add(tf.keras.layers.Conv2D(filters=32, kernel_size=(5, 5), activation="relu"))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2,2)))

model.add(tf.keras.layers.Conv2D(filters=64, kernel_size=(3, 3), activation="relu"))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2,2)))

# นำ layer สุดท้ายมาแผ่
model.add(tf.keras.layers.Flatten())

# Hidden Layer เพื่อทำพยากรณ์ผลลัพธ์
model.add(tf.keras.layers.Dense(200, activation="relu"))

# Dropout: ปิด neuron 25% ระหว่าง training เพื่อป้องกัน overfitting
model.add(tf.keras.layers.Dropout(0.25))

model.add(tf.keras.layers.Dense(100, activation="relu"))
model.add(tf.keras.layers.Dense(50, activation="relu"))
model.add(tf.keras.layers.Dense(25, activation="relu"))

# พยากรณ์ผลลัพธ์
model.add(tf.keras.layers.Dense(10, activation="softmax"))

model.summary()

In [ ]:
# Training
batch_size = 10000
epoch = 10

model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

history = model.fit(
  x_train,
  y_train,
  batch_size = batch_size,
  epochs = epoch,
  validation_split = 0.2
  # validation_data = (x_test, y_test)
)